# Batch layer trên Google Colab — CSE-CIC-IDS2018 với PySpark

Notebook chạy toàn bộ batch layer: ETL + báo cáo chất lượng dữ liệu → kịch bản 1 (chia ngẫu nhiên, trước/sau khử trùng lặp) → RF đa lớp → kịch bản 2 (leave-one-day-out) → benchmark mở rộng → biểu đồ.

**Chuẩn bị:** chép thư mục dự án (ít nhất `src/`) lên Google Drive, ví dụ `MyDrive/ie212/`. Dữ liệu CSV có thể để trong `MyDrive/ie212/data/` hoặc tải thẳng từ AWS ở ô bên dưới.

Kết quả (`results/`) và mô hình (`models/`) được ghi lên Drive; Parquet trung gian ghi vào đĩa cục bộ của Colab (`/content/work`) cho nhanh. Sau khi chạy xong, tải `models/` và `results/` về máy, đặt vào thư mục dự án để chạy speed/serving layer bằng Docker.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/ie212'   # <-- sửa nếu để chỗ khác
WORK = '/content/work'
import os; os.makedirs(WORK, exist_ok=True)
print(os.listdir(PROJECT))

In [ ]:
# Java 17 + PySpark 4.0.1 (cùng phiên bản với image Docker để mô hình đã lưu nạp được ở speed layer)
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark==4.0.1 pyarrow scikit-learn psutil matplotlib
import os, sys
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYTHONPATH'] = f"{PROJECT}/src"
os.environ['SPARK_DRIVER_MEMORY'] = '9g'      # Colab thường có ~12.7 GB RAM
sys.path.insert(0, f"{PROJECT}/src")
!java -version 2>&1 | head -1
!nproc; free -g | head -2

### (Tuỳ chọn) Tải dữ liệu từ AWS Open Data — không cần tài khoản
Bỏ qua nếu đã có 10 file CSV trong `PROJECT/data`. Tải về đĩa Colab (`/content/data`) nhanh hơn ghi lên Drive.

In [ ]:
DATA = f"{PROJECT}/data"
if not os.path.exists(DATA) or len([f for f in os.listdir(DATA) if f.endswith('.csv')]) < 10:
    !pip -q install awscli
    !aws s3 cp --no-sign-request "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/" /content/data --recursive
    DATA = '/content/data'
print(DATA, sorted(os.listdir(DATA)))

## 1. ETL + báo cáo chất lượng dữ liệu (≈16 triệu luồng)

In [ ]:
!cd "{PROJECT}" && python -m ids.etl --input "{DATA}" --output {WORK}/parquet --report results/quality_report.json

## 2. Kịch bản 1 — chia ngẫu nhiên phân tầng, có trùng lặp vs đã khử trùng lặp
`--train-frac` lấy mẫu phân tầng tập train (nhãn hiếm giữ nguyên) để vừa thời gian của Colab; tập validation/test giữ đầy đủ. Đặt `1.0` nếu có máy mạnh. Bước này cũng chọn mô hình có chi phí validation nhỏ nhất (tỷ lệ 1:50) và lưu vào `models/`.

In [ ]:
!cd "{PROJECT}" && python -m ids.train scenario1 --parquet {WORK}/parquet --work {WORK} --results results --models-out models --preset full --train-frac 0.3

## 3. Random Forest đa lớp (7 họ tấn công)

In [ ]:
!cd "{PROJECT}" && python -m ids.train multiclass --parquet {WORK}/parquet --work {WORK} --results results --models-out models --preset full --train-frac 0.3

## 4. Kịch bản 2 — leave-one-day-out
40 lần huấn luyện (10 ngày × 4 mô hình). `--resume` bỏ qua các fold đã xong nên có thể chạy lại ô này nếu Colab bị ngắt.

In [ ]:
!cd "{PROJECT}" && python -m ids.train scenario2 --parquet {WORK}/parquet --work {WORK} --results results --preset fast --train-frac 0.2 --resume

## 5. Benchmark khả năng mở rộng: Spark vs pandas/scikit-learn
Mẫu dữ liệu thật 25% / 50% / 100% (không nhân bản dữ liệu giả). Colab chỉ có 2 core; để đo đến 8 core hãy chạy cùng lệnh bằng dịch vụ `batch` trong Docker trên máy nhiều core.

In [ ]:
!cd "{PROJECT}" && python -m ids.bench prepare --data "{DATA}" --parquet {WORK}/parquet --bench {WORK}/bench
!cd "{PROJECT}" && python -m ids.bench all --data "{DATA}" --bench {WORK}/bench --cores 1,2 --fracs 0.25,0.5,1.0 --timeout 3600 --spark-mem 8g --note "Colab" --out results/scalability.json --resume

## 6. Biểu đồ + bảng tóm tắt cho báo cáo

In [ ]:
!cd "{PROJECT}" && python -m ids.figures --results results
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{PROJECT}/results/summary.md", encoding='utf-8').read()))
for f in sorted(os.listdir(f"{PROJECT}/results/figures")):
    display(Image(f"{PROJECT}/results/figures/{f}"))